In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

## LLM
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),          # deepseek-v4-flash
    api_key=os.getenv("EVREN_API_KEY"),     
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0    # https://evren-llmapi.ssyz.org.tr/v1
)   


## llm Test
response = llm.invoke([HumanMessage(content="Who is Atatürk?!")])
print("LLM Response:", response.content)

LLM Response: Mustafa Kemal Atatürk (1881–1938) was a Turkish military officer, revolutionary statesman, and the founder of the Republic of Turkey. He served as Turkey’s first President from 1923 until his death in 1938.

He is best known for leading Turkey’s War of Independence (1919–1923) after the collapse of the Ottoman Empire, and for launching sweeping political, social, legal, and cultural reforms that transformed Turkey into a modern, secular, and Western-oriented nation-state. His reforms included:

- Abolishing the Ottoman sultanate and caliphate
- Establishing a republic and a new legal system
- Adopting the Latin alphabet for Turkish
- Secularizing education and government
- Granting women equal civil and political rights (including the right to vote and be elected)

Atatürk’s legacy is deeply respected in Turkey, where he is commonly referred to as the “Father of the Turks” — the surname “Atatürk” was given to him by the Turkish parliament in 1934. His principles, known as

In [2]:
import redis
import hashlib
import json
import time
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

# ─────────────────────────────────────────────
#  REDIS CONNECTION
# ─────────────────────────────────────────────
r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    decode_responses=True
)

# ─────────────────────────────────────────────
#  LLM INITIALIZATION (via Evren API)
# ─────────────────────────────────────────────
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),
    api_key=os.getenv("EVREN_API_KEY"),
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0
)

# ─────────────────────────────────────────────
#  HEALTH CHECK
# ─────────────────────────────────────────────
r.ping()

print("\n" + "═" * 58)
print("\033[1;36m  🚀  SYSTEM INITIALIZATION\033[0m")
print("═" * 58)

redis_status = "\033[1;32m✅ CONNECTED\033[0m" if r.ping() else "\033[1;31m❌ FAILED\033[0m"
print(f"  \033[1;33m▸ Redis\033[0m            → {redis_status}")
print(f"     \033[90mhost:\033[0m {os.getenv('REDIS_HOST', 'localhost')}:{os.getenv('REDIS_PORT', 6379)}")

api_status = "\033[1;32m✅ LOADED\033[0m" if os.getenv("EVREN_API_KEY") else "\033[1;31m❌ MISSING\033[0m"
print(f"  \033[1;33m▸ Evren API Key\033[0m    → {api_status}")

model_name = os.getenv("EVREN_MODEL", "unknown")
print(f"  \033[1;33m▸ LLM Model\033[0m        → \033[1;32m{model_name}\033[0m")
print(f"     \033[90mbase_url:\033[0m {os.getenv('EVREN_BASE_URL', 'N/A')}")

print("═" * 58 + "\n")


══════════════════════════════════════════════════════════
  🚀  SYSTEM INITIALIZATION
══════════════════════════════════════════════════════════
  ▸ Redis            → ✅ CONNECTED
     host: localhost:6379
  ▸ Evren API Key    → ✅ LOADED
  ▸ LLM Model        → deepseek-v4-flash
     base_url: https://evren-llmapi.ssyz.org.tr/v1
══════════════════════════════════════════════════════════



In [3]:
# ─────────────────────────────────────────────────
# DEMO: What MD5 hashing looks like
# Same input ALWAYS produces the same hash
# ─────────────────────────────────────────────────

def make_cache_key(prompt: str) -> str:
    # Normalize: strip whitespace, lowercase — so "Hello" and "hello " both map to the same key
    normalized = prompt.strip().lower()
    hash_value = hashlib.md5(normalized.encode()).hexdigest()
    return f"llm:cache:{hash_value}"

# Show how keys are generated
test_prompts = [
    "What is machine learning?",   # baseline
    "What is machine learning?",   # identical — same key
    "What is machine learning? ",  # trailing space — normalized away, still same key
    "What is deep learning?",      # different prompt — different key
]

# ─────────────────────────────────────────────────
# OUTPUT
# ─────────────────────────────────────────────────
print("\n" + "═" * 72)
print("\033[1;36m  🔑  MD5 CACHE KEY GENERATION\033[0m")
print("\033[90m  Normalized (strip + lowercase) → MD5 hash → Redis key\033[0m")
print("═" * 72)

print(f"\n  \033[1;33m{'PROMPT':<38}\033[0m \033[1;33m{'CACHE KEY'}\033[0m")
print("  " + "─" * 68)

seen_keys = {}
for p in test_prompts:
    key = make_cache_key(p)
    short_prompt = p[:35] + ("..." if len(p) > 35 else "")

    
    if key in seen_keys:
        color = "\033[1;33m"   # yellow = duplicate
        marker = "↺"
    else:
        color = "\033[1;32m"   # green = unique
        marker = "✦"
        seen_keys[key] = p

    print(f"  {marker} {color}{short_prompt:<38}\033[0m \033[90m{key}\033[0m")

unique_count = len(seen_keys)
total_count = len(test_prompts)

print("\n  " + "─" * 68)
print(f"  \033[1;34m📊 Total prompts:\033[0m {total_count}   "
      f"\033[1;34mUnique keys:\033[0m \033[1;32m{unique_count}\033[0m   "
      f"\033[1;34mCache hits:\033[0m \033[1;33m{total_count - unique_count}\033[0m")
print("═" * 72)

print(f"\n  \033[90m💡 Note: The first three prompts produce the SAME key\033[0m")
print(f"  \033[90m   (exact-match caching via normalization).\033[0m\n")


════════════════════════════════════════════════════════════════════════
  🔑  MD5 CACHE KEY GENERATION
  Normalized (strip + lowercase) → MD5 hash → Redis key
════════════════════════════════════════════════════════════════════════

  PROMPT                                 CACHE KEY
  ────────────────────────────────────────────────────────────────────
  ✦ What is machine learning?              llm:cache:0c9fdf1c9bd767b44887ab8491c4b9c0
  ↺ What is machine learning?              llm:cache:0c9fdf1c9bd767b44887ab8491c4b9c0
  ↺ What is machine learning?              llm:cache:0c9fdf1c9bd767b44887ab8491c4b9c0
  ✦ What is deep learning?                 llm:cache:28a3b39d74471f87bd1ec238a0efe079

  ────────────────────────────────────────────────────────────────────
  📊 Total prompts: 4   Unique keys: 2   Cache hits: 2
════════════════════════════════════════════════════════════════════════

  💡 Note: The first three prompts produce the SAME key
     (exact-match caching via normalization).

In [6]:
import json
import time
import hashlib
import os


MODEL_NAME = os.getenv("EVREN_MODEL", "deepseek-v4-flash")

COST_PER_INPUT_TOKEN  = 0.00000015
COST_PER_OUTPUT_TOKEN = 0.00000060

def make_cache_key(prompt: str) -> str:
    """Prompt için standart Redis cache anahtarı üretir (SHA256)."""
    cleaned = prompt.strip().lower()
    hash_val = hashlib.sha256(cleaned.encode("utf-8")).hexdigest()
    return f"llm:cache:{hash_val}"

def get_from_cache(prompt: str):
    """Redis üzerinde önbellek varsa dict döner, yoksa None."""
    key = make_cache_key(prompt)
    raw = r.get(key)
    if raw:
        return json.loads(raw)
    return None

def save_to_cache(prompt: str, response: str, token_info: dict, ttl_seconds: int = 3600):
    """LLM cevabını ve meta verilerini Redis'e kaydeder."""
    key = make_cache_key(prompt)
    
   
    model_val = token_info.get("model", MODEL_NAME)
    if hasattr(model_val, "model_name"):
        model_str = model_val.model_name
    elif not isinstance(model_val, str):
        model_str = str(model_val)
    else:
        model_str = model_val

    data = {
        "prompt": prompt,
        "response": response,
        "model": model_str,
        "input_tokens": int(token_info.get("input_tokens", 0)),
        "output_tokens": int(token_info.get("output_tokens", 0)),
        "cached_at": time.time()
    }
    
    r.set(key, json.dumps(data), ex=ttl_seconds)

def ask_with_cache(prompt: str, ttl: int = 3600) -> dict:
    """Prompt'u önbellekten veya LLM'den getirir."""
    start = time.time()

    # 1. Adım: Cache Control
    cached = get_from_cache(prompt)
    if cached:
        elapsed_ms = (time.time() - start) * 1000
        return {
            "response": cached["response"],
            "source": "CACHE HIT",
            "time_ms": round(elapsed_ms, 2),
            "cost_usd": 0.0,
            "input_tokens": 0,
            "output_tokens": 0,
        }

    # 2. Adım: Cache Miss — LLM Call
    ai_msg = llm.invoke([HumanMessage(content=prompt)])
    elapsed_ms = (time.time() - start) * 1000

    response_text = ai_msg.content
    
    # Token info
    usage = getattr(ai_msg, "usage_metadata", {}) or {}
    input_tokens = usage.get("input_tokens", 0)
    output_tokens = usage.get("output_tokens", 0)
    
    cost = (input_tokens * COST_PER_INPUT_TOKEN) + (output_tokens * COST_PER_OUTPUT_TOKEN)

    # 3. Adım: Save the cache
    save_to_cache(
        prompt=prompt,
        response=response_text,
        token_info={
            "model": MODEL_NAME, 
            "input_tokens": input_tokens,
            "output_tokens": output_tokens
        },
        ttl_seconds=ttl
    )

    return {
        "response": response_text,
        "source": "API CALL",
        "time_ms": round(elapsed_ms, 2),
        "cost_usd": round(cost, 8),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

print("Cache functions defined!")


Cache functions defined!


## Cache Hit vs Cache Miss

In [7]:
prompt = "What is the difference between supervised and unsupervised machine learning?"

print("\n" + "═" * 62)
print("\033[1;36m  🌐  FIRST CALL  \033[0m\033[90m(expected: API CALL)\033[0m")
print("═" * 62)

result1 = ask_with_cache(prompt)

source = result1['source']
if source == "CACHE HIT":
    source_display = "\033[1;32m⚡ CACHE HIT\033[0m"
elif source == "API CALL":
    source_display = "\033[1;33m🌐 API CALL\033[0m"
else:
    source_display = f"\033[1;37m{source}\033[0m"

print(f"\n  \033[1;34m▸ Source\033[0m     → {source_display}")
print(f"  \033[1;34m▸ Time\033[0m       → \033[1;32m{result1['time_ms']} ms\033[0m")
print(f"  \033[1;34m▸ Cost\033[0m       → \033[1;33m${result1['cost_usd']}\033[0m")
print(f"  \033[1;34m▸ Tokens\033[0m     → \033[90m{result1['input_tokens']} in\033[0m / "
      f"\033[90m{result1['output_tokens']} out\033[0m")

print(f"\n  \033[1;34m▸ Response\033[0m")
print(f"     \033[90m{result1['response'][:150]}...\033[0m")

print("\n" + "═" * 62 + "\n")


══════════════════════════════════════════════════════════════
  🌐  FIRST CALL  (expected: API CALL)
══════════════════════════════════════════════════════════════

  ▸ Source     → 🌐 API CALL
  ▸ Time       → 1824.38 ms
  ▸ Cost       → $0.0002547
  ▸ Tokens     → 94 in / 401 out

  ▸ Response
     The main difference is whether the algorithm learns from **labeled data** (with known answers) or **unlabeled data** (without answers).

### Supervise...

══════════════════════════════════════════════════════════════



In [8]:
print("\n" + "═" * 62)
print("\033[1;36m  ⚡  SECOND CALL  \033[0m\033[90m(same prompt — expected: CACHE HIT)\033[0m")
print("═" * 62)

result2 = ask_with_cache(prompt)

# Determine color and icon based on source
source2 = result2['source']
if source2 == "CACHE HIT":
    source_display2 = "\033[1;32m⚡ CACHE HIT\033[0m"
elif source2 == "API CALL":
    source_display2 = "\033[1;33m🌐 API CALL\033[0m"
else:
    source_display2 = f"\033[1;37m{source2}\033[0m"

print(f"\n  \033[1;34m▸ Source\033[0m     → {source_display2}")
print(f"  \033[1;34m▸ Time\033[0m       → \033[1;32m{result2['time_ms']} ms\033[0m")
print(f"  \033[1;34m▸ Cost\033[0m       → \033[1;33m${result2['cost_usd']}\033[0m")

print(f"\n  \033[1;34m▸ Response\033[0m")
print(f"     \033[90m{result2['response'][:150]}...\033[0m")

# ─────────────────────────────────────────────
# SPEEDUP COMPARISON
# ─────────────────────────────────────────────
if result1["time_ms"] > 0 and result2["time_ms"] > 0:
    speedup = result1["time_ms"] / result2["time_ms"]

    print("\n" + "─" * 62)
    print("\033[1;35m  🚀  PERFORMANCE COMPARISON\033[0m")
    print("─" * 62)

    print(f"\n  \033[90mAPI Call (first):\033[0m   \033[1;33m{result1['time_ms']:>10.2f} ms\033[0m")
    print(f"  \033[90mCache Hit (second):\033[0m \033[1;32m{result2['time_ms']:>10.2f} ms\033[0m")

    print(f"\n  \033[1;32m✅ Cache was \033[1;37m{speedup:.0f}x\033[1;32m faster than the API call!\033[0m")
    print(f"  \033[90m💰 Cost saved: ${result1['cost_usd']}\033[0m")

print("\n" + "═" * 62 + "\n")


══════════════════════════════════════════════════════════════
  ⚡  SECOND CALL  (same prompt — expected: CACHE HIT)
══════════════════════════════════════════════════════════════

  ▸ Source     → ⚡ CACHE HIT
  ▸ Time       → 8.96 ms
  ▸ Cost       → $0.0

  ▸ Response
     The main difference is whether the algorithm learns from **labeled data** (with known answers) or **unlabeled data** (without answers).

### Supervise...

──────────────────────────────────────────────────────────────
  🚀  PERFORMANCE COMPARISON
──────────────────────────────────────────────────────────────

  API Call (first):      1824.38 ms
  Cache Hit (second):       8.96 ms

  ✅ Cache was 204x faster than the API call!
  💰 Cost saved: $0.0002547

══════════════════════════════════════════════════════════════



In [9]:
# ─────────────────────────────────────────────
# DEMO: Normalization — slightly different
# phrasing of the SAME question also hits cache
# because we strip and lowercase before hashing
# ─────────────────────────────────────────────

print("\n" + "═" * 72)
print("\033[1;36m  🔤  NORMALIZATION DEMO\033[0m")
print("\033[90m  Testing trim + lowercase — variants of the same question\033[0m")
print("═" * 72)

# These should all hit the cache from the call above
# because they normalize to the same string
variants = [
    "What is the difference between supervised and unsupervised machine learning?",
    "  What is the difference between supervised and unsupervised machine learning?  ",
    "WHAT IS THE DIFFERENCE BETWEEN SUPERVISED AND UNSUPERVISED MACHINE LEARNING?",
]

print(f"\n  \033[1;33m{'VARIANT':<58}\033[0m \033[1;33m{'RESULT'}\033[0m")
print("  " + "─" * 68)

hits = 0
misses = 0

for v in variants:
    result = ask_with_cache(v)
    short_prompt = v.strip()[:55] + "..."

    # Determine color and icon based on source
    if result['source'] == "CACHE HIT":
        source_display = "\033[1;32m⚡ CACHE HIT\033[0m"
        hits += 1
    else:
        source_display = "\033[1;33m🌐 API CALL\033[0m"
        misses += 1

    print(f"  \033[90m▸\033[0m {short_prompt:<58} {source_display}  \033[90m({result['time_ms']} ms)\033[0m")

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
print("\n  " + "─" * 68)
print(f"  \033[1;34m📊 Variants tested:\033[0m {len(variants)}   "
      f"\033[1;34mCache hits:\033[0m \033[1;32m{hits}\033[0m   "
      f"\033[1;34mAPI calls:\033[0m \033[1;33m{misses}\033[0m")
print("═" * 72)

print(f"\n  \033[90m💡 Note: All variants normalize (strip + lowercase) to the\033[0m")
print(f"  \033[90m   same string, so they all hit the SAME cache entry.\033[0m\n")


════════════════════════════════════════════════════════════════════════
  🔤  NORMALIZATION DEMO
  Testing trim + lowercase — variants of the same question
════════════════════════════════════════════════════════════════════════

  VARIANT                                                    RESULT
  ────────────────────────────────────────────────────────────────────
  ▸ What is the difference between supervised and unsupervi... ⚡ CACHE HIT  (11.98 ms)
  ▸ What is the difference between supervised and unsupervi... ⚡ CACHE HIT  (1.77 ms)
  ▸ WHAT IS THE DIFFERENCE BETWEEN SUPERVISED AND UNSUPERVI... ⚡ CACHE HIT  (0.9 ms)

  ────────────────────────────────────────────────────────────────────
  📊 Variants tested: 3   Cache hits: 3   API calls: 0
════════════════════════════════════════════════════════════════════════

  💡 Note: All variants normalize (strip + lowercase) to the
     same string, so they all hit the SAME cache entry.

